

`Práctica Nº 7 - Aprendizaje Profundo`



#**CLASIFICACIÓN DE TEXTO CON TRANSFORMERS**

##*DataSet: poem_sentiment*

### 00. Librerias

In [ ]:
!pip install -q datasets evaluate transformers[sentencepiece] accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np
import pandas as pd

### 01. DataSet

El dataset usado contiene poemas etiquetados en 4 clases:
- 0 = negative
- 1 = positive
- 2 = no impact
- 3 = mixed

In [ ]:
dataset = load_dataset("poem_sentiment")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/6.34k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/6.16k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/892 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/105 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/104 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 892
    })
    validation: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 105
    })
    test: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 104
    })
})


In [ ]:
print("\nEjemplo del train:")
print(dataset["train"][0])


Ejemplo del train:
{'id': 0, 'verse_text': 'with pale blue berries. in these peaceful shades--', 'label': 1}


In [ ]:
print("\nFeatures:")
print(dataset["train"].features)


Features:
{'id': Value('int32'), 'verse_text': Value('string'), 'label': ClassLabel(names=['negative', 'positive', 'no_impact', 'mixed'])}




> Conversión a DataFrame


In [ ]:
train_df = dataset["train"].to_pandas()
valid_df = dataset["validation"].to_pandas()
test_df  = dataset["test"].to_pandas()

In [ ]:
print("\nPrimeras filas del train:")
display(train_df.head())


Primeras filas del train:


,id,verse_text,label
0,0,with pale blue berries. in these peaceful shad...,1
1,1,"it flows so long as falls the rain,",2
2,2,"and that is why, the lonesome day,",0
3,3,"when i peruse the conquered fame of heroes, an...",3
4,4,of inward strife for truth and liberty.,3


In [ ]:
print("\nTamaño train:", len(train_df))
print("Tamaño validation:", len(valid_df))
print("Tamaño test:", len(test_df))


Tamaño train: 892
Tamaño validation: 105
Tamaño test: 104




> Definición de etiquetas



In [ ]:
label_names = dataset["train"].features["label"].names
print("\nNombres de clases:", label_names)


Nombres de clases: ['negative', 'positive', 'no_impact', 'mixed']


In [ ]:
id2label = {i: label_names[i] for i in range(len(label_names))}
label2id = {label_names[i]: i for i in range(len(label_names))}

In [ ]:
print("id2label:", id2label)
print("label2id:", label2id)

id2label: {0: 'negative', 1: 'positive', 2: 'no_impact', 3: 'mixed'}
label2id: {'negative': 0, 'positive': 1, 'no_impact': 2, 'mixed': 3}


###02. Tokenización

In [ ]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def tokenize_function(batch):
    # Tokenizamos la columna de texto del dataset.
    # truncation=True evita secuencias demasiado largas.
    return tokenizer(batch["verse_text"], truncation=True)

In [ ]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/892 [00:00<?, ? examples/s]

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

In [ ]:
print("\nDataset tokenizado:")
print(tokenized_dataset)


Dataset tokenizado:
DatasetDict({
    train: Dataset({
        features: ['id', 'verse_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 892
    })
    validation: Dataset({
        features: ['id', 'verse_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 105
    })
    test: Dataset({
        features: ['id', 'verse_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 104
    })
})


###03. Data Collator

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

###04. Métrica de evaluación

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

In [ ]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds

    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)

    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")

    return {
        "accuracy": acc["accuracy"],
        "f1_macro": f1["f1"]
    }

### 05. Modelo preentrenado

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="clasificador-poem-sentiment-bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none"
)


###06. Entrenamiento



In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.075014,0.832982,0.666667,0.299229
2,0.783800,0.576487,0.838095,0.731118
3,0.573336,0.538477,0.838095,0.736713


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.075014,0.832982,0.666667,0.299229
2,0.783800,0.576487,0.838095,0.731118
3,0.573336,0.538477,0.838095,0.736713


TrainOutput(global_step=168, training_loss=0.810716606321789, metrics={'train_runtime': 911.7471, 'train_samples_per_second': 2.935, 'train_steps_per_second': 0.184, 'total_flos': 26399968466976.0, 'train_loss': 0.810716606321789, 'epoch': 3.0})

###07. Evaluación sobre validation

In [ ]:
val_results = trainer.evaluate(tokenized_dataset["validation"])
print("\nResultados en validation:")
print(val_results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Resultados en validation:
{'eval_loss': 0.576326847076416, 'eval_accuracy': 0.8380952380952381, 'eval_f1_macro': 0.7311176278918214, 'eval_runtime': 13.8478, 'eval_samples_per_second': 7.582, 'eval_steps_per_second': 0.505, 'epoch': 3.0}


###08. Evaluación sobre test

In [ ]:
test_results = trainer.evaluate(tokenized_dataset["test"])
print("\nResultados en test:")
print(test_results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Resultados en test:
{'eval_loss': 0.5376404523849487, 'eval_accuracy': 0.8365384615384616, 'eval_f1_macro': 0.7469928346788576, 'eval_runtime': 7.7903, 'eval_samples_per_second': 13.35, 'eval_steps_per_second': 0.899, 'epoch': 3.0}


###09. Predicción sobre textos

In [ ]:
import torch

sample_texts = [
    "the sky smiles softly over the valley",
    "my heart breaks in silence and cold ashes",
    "the window is open and the table is made of wood",
    "joy and sorrow dance together in the same breath"
]


inputs = tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)


model.eval()


with torch.no_grad():
    outputs = model(**inputs)


pred_ids = torch.argmax(outputs.logits, dim=1)


for text, pred_id in zip(sample_texts, pred_ids):
    print("Texto:", text)
    print("Clase predicha:", id2label[pred_id.item()])
    print("-" * 50)

Texto: the sky smiles softly over the valley
Clase predicha: no_impact
--------------------------------------------------
Texto: my heart breaks in silence and cold ashes
Clase predicha: negative
--------------------------------------------------
Texto: the window is open and the table is made of wood
Clase predicha: no_impact
--------------------------------------------------
Texto: joy and sorrow dance together in the same breath
Clase predicha: positive
--------------------------------------------------


###10. Guardar modelo

In [ ]:
trainer.save_model("modelo_final_poem_sentiment_bert")
tokenizer.save_pretrained("modelo_final_poem_sentiment_bert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('modelo_final_poem_sentiment_bert/tokenizer_config.json',
 'modelo_final_poem_sentiment_bert/tokenizer.json')

In [ ]:
print("\nModelo guardado correctamente.")


Modelo guardado correctamente.
